# Week 2: Corpus Collection & Initial Exploration

This notebook inventories the filtered academic files stored under `data/raw/s1` through `data/raw/s8` (`.pdf`, `.docx`, `.pptx`, and `.ipynb`) to support the **Week 2: Data Collection & Understanding** deliverable (`LastName-w3.pdf`).

### Goals
- Count files and total size by semester and file format
- Inspect metadata fields (semester, inferred category, file path, size)
- Estimate language mix (French vs. English) and text/length volume
- Flag potential data quality issues (empty/unreadable files, low-text/scanned PDFs, heavy slide decks or notebooks) before text cleaning in Week 3

In [5]:
from pathlib import Path
import json
import re
import pandas as pd
import langdetect

# Point to your local raw corpus folder (ignored by Git via .gitignore)
RAW_DATA_DIR = Path("../data/raw")
TARGET_EXTENSIONS = {".pdf", ".docx", ".pptx", ".ipynb"}

# Optional lightweight libraries for quick text/language inspection
try:
    import fitz  # PyMuPDF
except ImportError:
    fitz = None

try:
    from docx import Document
except ImportError:
    Document = None

try:
    from pptx import Presentation
except ImportError:
    Presentation = None

try:
    from langdetect import detect, LangDetectException
except ImportError:
    detect = None
    LangDetectException = Exception

print("RAW_DATA_DIR exists:", RAW_DATA_DIR.exists())
print("Available optional packages -> PyMuPDF:", bool(fitz), "| python-docx:", bool(Document), "| python-pptx:", bool(Presentation), "| langdetect:", bool(detect))

RAW_DATA_DIR exists: True
Available optional packages -> PyMuPDF: True | python-docx: True | python-pptx: True | langdetect: True


## 1. File Inventory & Metadata Extraction

We walk `data/raw/` recursively for target extensions (`.pdf`, `.docx`, `.pptx`, `.ipynb`) and record basic metadata: semester folder, inferred document category from path/filename keywords, file size, and relative path.

In [9]:
def infer_semester(rel_parts: tuple[str, ...]) -> str:
    for part in rel_parts:
        match = re.fullmatch(r"[sS]([1-9])", part)
        if match:
            return f"S{match.group(1)}"
    return "Unknown"


def infer_doc_type(rel_path_str: str) -> str:
    ext = Path(rel_path_str).suffix.lower()
    mapping = {
        ".pdf": "pdf_document",
        ".docx": "word_document",
        ".pptx": "slide_deck",
        ".ipynb": "jupyter_notebook",
    }
    return mapping.get(ext, "other")


records = []
if RAW_DATA_DIR.exists():
    for path in RAW_DATA_DIR.rglob("*"):
        if not path.is_file():
            continue
        ext = path.suffix.lower()
        if ext not in TARGET_EXTENSIONS:
            continue

        rel_path = path.relative_to(RAW_DATA_DIR)
        size_bytes = path.stat().st_size
        records.append({
            "file_name": path.name,
            "extension": ext,
            "semester": infer_semester(rel_path.parts),
            "doc_type": infer_doc_type(str(rel_path)),
            "size_bytes": size_bytes,
            "size_mb": round(size_bytes / (1024 * 1024), 3),
            "rel_path": str(rel_path),
        })

df_files = pd.DataFrame(records)
print(f"Total matching files found: {len(df_files)}")
if not df_files.empty:
    print(f"Total size: {df_files['size_mb'].sum():.2f} MB ({df_files['size_mb'].sum() / 1024:.2f} GB)")
    display(df_files.head(3))

Total matching files found: 620
Total size: 1285.65 MB (1.26 GB)


,file_name,extension,semester,doc_type,size_bytes,size_mb,rel_path
0,exposés_MIPS_25-26.pdf,.pdf,S8,pdf_document,38581,0.037,S8\Business Intelligence\exposés_MIPS_25-26.pdf
1,Integration_donnees.pdf,.pdf,S8,pdf_document,1409173,1.344,S8\Business Intelligence\Integration_donnees.pdf
2,synthese_integration_donnees_v2_hayyani.pdf,.pdf,S8,pdf_document,1718944,1.639,S8\Business Intelligence\synthese_integration_...


## 2. Distribution by Semester, Format, and Document Type

Here we summarize how many files and how much disk space belong to each semester (`S1`–`S8`), format, and inferred document type.

In [10]:
if not df_files.empty:
    by_semester_ext = pd.crosstab(
        index=df_files["semester"],
        columns=df_files["extension"],
        margins=True,
    )
    print("File counts by semester and extension:")
    display(by_semester_ext)

    size_by_semester = (
        df_files.groupby("semester", dropna=False)["size_mb"]
        .agg(file_count="count", total_mb="sum", mean_mb="mean")
        .round(2)
    )
    print("\nSize summary (MB) by semester:")
    display(size_by_semester)

    print("\nCounts by inferred document type:")
    display(df_files["doc_type"].value_counts().to_frame("count"))

File counts by semester and extension:


extension,.docx,.ipynb,.pdf,.pptx,All
semester,,,,,
S1,6,0,22,4,32
S2,16,0,49,10,75
S3,11,0,46,9,66
S4,4,0,21,3,28
S5,35,40,53,6,134
S6,31,27,24,7,89
S7,17,6,85,4,112
S8,12,6,60,6,84
All,132,79,360,49,620



Size summary (MB) by semester:


,file_count,total_mb,mean_mb
semester,,,
S1,32,31.12,0.97
S2,75,116.31,1.55
S3,66,235.60,3.57
S4,28,66.80,2.39
S5,134,194.45,1.45
S6,89,72.62,0.82
S7,112,189.92,1.70
S8,84,378.84,4.51



Counts by inferred document type:


,count
doc_type,
pdf_document,360
word_document,132
jupyter_notebook,79
slide_deck,49


## 3. Sample Text Extraction, Language Mix & Quality Flags

To spot potential quality issues early—such as unreadable/corrupt files, scanned PDFs with almost no selectable text, empty notebooks, or mixed French/English content—we extract a quick sample of text from each file and flag edge cases for closer inspection.

In [12]:
def read_text(path: Path) -> tuple[str, str]:
    ext = path.suffix.lower()
    try:
        if path.stat().st_size == 0:
            return "", "empty_file"
        if ext == ".ipynb":
            cells = json.loads(path.read_text(encoding="utf-8", errors="replace")).get("cells", [])
            text = "\n".join("".join(c.get("source", [])) for c in cells)
        elif ext == ".pdf":
            with fitz.open(path) as doc:
                text = "\n".join(page.get_text("text") for page in doc)
        elif ext == ".docx":
            doc = Document(path)
            text = "\n".join(p.text for p in doc.paragraphs if p.text.strip())
        elif ext == ".pptx":
            prs = Presentation(path)
            text = "\n".join(
                sh.text_frame.text for sl in prs.slides for sh in sl.shapes if sh.has_text_frame
            )
        else:
            return "", "unsupported"
        return text, ("low_text" if len(text.strip()) < 100 else "ok")
    except Exception as e:
        return "", f"error:{type(e).__name__}"


def get_lang(text: str) -> str:
    clean = re.sub(r"\s+", " ", text).strip()
    if len(clean) < 40:
        return "short_or_empty"
    try:
        return detect(clean[:2000])
    except LangDetectException:
        return "unknown"


if not df_files.empty:
    texts, chars, langs, flags = [], [], [], []
    for rel_p in df_files["rel_path"]:
        txt, flag = read_text(RAW_DATA_DIR / rel_p)
        texts.append(re.sub(r"\s+", " ", txt)[:200])
        chars.append(len(txt))
        langs.append(get_lang(txt))
        flags.append(flag)

    df_files["chars"] = chars
    df_files["lang"] = langs
    df_files["flag"] = flags
    df_files["preview"] = texts

    display(df_files["lang"].value_counts().to_frame("files"))
    display(df_files["flag"].value_counts().to_frame("files"))
    display(df_files[df_files["flag"] != "ok"][["semester", "extension", "file_name", "chars", "flag"]])

MuPDF error: syntax error: too few subfunction boundaries

MuPDF error: syntax error: too few subfunction boundaries



,files
lang,
fr,485
en,109
short_or_empty,26


,files
flag,
ok,592
low_text,28


,semester,extension,file_name,chars,flag
22,S8,.pptx,cours-software-developement-1.pptx,80,low_text
58,S8,.pdf,Guide TP_2.pdf,14,low_text
61,S8,.pdf,Guide TP_3.pdf,14,low_text
81,S8,.pdf,presentation 3-1.pdf,40,low_text
125,S7,.pdf,Systèmes distribués basés sur les micro-servic...,16,low_text
126,S7,.pdf,Systèmes distribués basés sur les micro-servic...,28,low_text
127,S7,.pdf,Systèmes distribués basés sur les micro-servic...,21,low_text
128,S7,.pdf,Annexe 2.pdf,2,low_text
199,S6,.pptx,DAT.pptx,0,low_text
224,S6,.docx,Résumé_detail.docx,0,low_text


## 4. Summary Notes for Week 2 Report (`LastName-w3.pdf`)

- **Corpus coverage:** The filtered inventory contains **620 files** (**1,285.65 MB / 1.26 GB**) across **S1–S8**, consisting of **360 PDFs** (`pdf_document`), **132 Word files** (`word_document`), **79 Jupyter notebooks** (`jupyter_notebook`), and **49 slide decks** (`slide_deck`). Notebooks only appear starting in **S5** (40 in S5, 27 in S6, 6 in S7, 6 in S8), while **S5** has the highest file count (**134 files**, 194.45 MB) and **S8** has the largest total volume (**378.84 MB**, 4.51 MB mean).
- **Language profile:** Out of 620 files, **485 are French (`fr`)**, **109 are English (`en`)**, and **26 are `short_or_empty`**—confirming a predominantly French corpus with a solid English subset that preprocessing and embeddings will need to handle.
- **Data quality & next steps:** **592 files** passed with `ok` and **28 files** were flagged as `low_text` (< 100 extracted characters), including **4 completely empty (`0` chars)** files (`DAT.pptx`, `Résumé_detail.docx`, `hdfs-cheatsheet.pdf`, `seance 5-checkpoint.ipynb`) and multiple near-empty PDFs/slides/notebooks (likely scanned documents, image-heavy slides, or sparse files), alongside minor MuPDF syntax warnings on certain PDFs. In Week 3, these `low_text` files should be inspected/filtered or selectively OCR'd where needed before chunking and indexing.